In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('../data/WA_Fn-UseC_-HR-Employee-Attrition.csv')
print(df.head(10))

   Age Attrition     BusinessTravel  DailyRate              Department  \
0   41       Yes      Travel_Rarely       1102                   Sales   
1   49        No  Travel_Frequently        279  Research & Development   
2   37       Yes      Travel_Rarely       1373  Research & Development   
3   33        No  Travel_Frequently       1392  Research & Development   
4   27        No      Travel_Rarely        591  Research & Development   
5   32        No  Travel_Frequently       1005  Research & Development   
6   59        No      Travel_Rarely       1324  Research & Development   
7   30        No      Travel_Rarely       1358  Research & Development   
8   38        No  Travel_Frequently        216  Research & Development   
9   36        No      Travel_Rarely       1299  Research & Development   

   DistanceFromHome  Education EducationField  EmployeeCount  EmployeeNumber  \
0                 1          2  Life Sciences              1               1   
1                 8      

In [3]:
df = df.drop(columns=['EmployeeCount', 'StandardHours', 'Over18'])

In [4]:
print(df.columns.tolist())

['Age', 'Attrition', 'BusinessTravel', 'DailyRate', 'Department', 'DistanceFromHome', 'Education', 'EducationField', 'EmployeeNumber', 'EnvironmentSatisfaction', 'Gender', 'HourlyRate', 'JobInvolvement', 'JobLevel', 'JobRole', 'JobSatisfaction', 'MaritalStatus', 'MonthlyIncome', 'MonthlyRate', 'NumCompaniesWorked', 'OverTime', 'PercentSalaryHike', 'PerformanceRating', 'RelationshipSatisfaction', 'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance', 'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager']


In [5]:
faixas = [18, 30, 40, 50, 100]
labels = ['18-30', '31-40', '41-50', '51+']

df['FaixaEtaria'] = pd.cut(df['Age'], bins=faixas, labels=labels, right=False)

print(df['FaixaEtaria'].value_counts())


FaixaEtaria
31-40    622
41-50    349
18-30    326
51+      173
Name: count, dtype: int64


In [6]:
print(df['MonthlyIncome'].describe())

count     1470.000000
mean      6502.931293
std       4707.956783
min       1009.000000
25%       2911.000000
50%       4919.000000
75%       8379.000000
max      19999.000000
Name: MonthlyIncome, dtype: float64


In [7]:
faixas = [1000, 3001, 6001, 10001, 20000]
labels = ['Low', 'Medium', 'High', 'Very High']

df['FaixaSalarial'] = pd.cut(df['MonthlyIncome'], bins=faixas, labels=labels, right=False)

print(df['FaixaSalarial'].value_counts())

FaixaSalarial
Medium       519
Low          395
Very High    281
High         275
Name: count, dtype: int64


In [8]:
df['AttritionBinario'] = df['Attrition'].map({'Yes': 1, 'No': 0})

In [9]:
print(df['AttritionBinario'].value_counts())

AttritionBinario
0    1233
1     237
Name: count, dtype: int64


In [10]:
faixas = [1, 11, 21, 30]
labels = ['Near', 'Medium', 'Far']

df['FaixaDistancia'] = pd.cut(df['DistanceFromHome'], bins=faixas, labels=labels, right=False)

print(df['FaixaDistancia'].value_counts())

FaixaDistancia
Near      1026
Medium     240
Far        204
Name: count, dtype: int64


In [11]:
faixas = [0, 3, 6, 11, 100]
labels = ['0-2 years', '3-5 years', '6-10 years', '11+ years']

df['FaixaTempoEmpresa'] = pd.cut(df['YearsAtCompany'], bins=faixas, labels=labels, right=False)

print(df['FaixaTempoEmpresa'].value_counts())

FaixaTempoEmpresa
6-10 years    448
3-5 years     434
0-2 years     342
11+ years     246
Name: count, dtype: int64


In [12]:
df.to_csv('../data/hr_tratado.csv', index=False)

In [13]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv()
server = os.getenv('DB_SERVER')

engine = create_engine(
    f'mssql+pyodbc://{server}/hr_analytics'
    '?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes'
)

df['FaixaEtaria'] = df['FaixaEtaria'].astype(str)
df['FaixaSalarial'] = df['FaixaSalarial'].astype(str)
df['FaixaDistancia'] = df['FaixaDistancia'].astype(str)
df['FaixaTempoEmpresa'] = df['FaixaTempoEmpresa'].astype(str)

df.to_sql('funcionarios', con=engine, if_exists='replace', index=False)

14